<a href="https://colab.research.google.com/github/Digitalbuilder2026/Intro-to-Computer-Vision/blob/main/real_book_symbolic_analysis_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Real Book Lead-Sheet Analyzer — Symbolic Music Theory (v2)

**Goal:** Upload a printed lead-sheet PDF, convert it to **MusicXML** with optical music recognition (OMR), verify the transcription, then analyze chord symbols and the written melody with `music21`.

**Honest limitation:** A scanned PDF is a picture, **not** a symbolic music file. `music21` does **not** read notes from scanned PDFs. Audiveris handles OMR, but its results require review—particularly jazz chord symbols, accidentals, repeats, ties, and unusual typography. This notebook provides the PDF upload and preview, **clear external OMR handoff**, and a working symbolic analyzer once you upload `.mxl`, `.musicxml`, or `.xml`. The OMR stage is **not fully automatic in this version**.

Use scores you are authorized to access; don't redistribute copyrighted Real Book pages or converted scores.

### Pipeline
`PDF → view/check pages → Audiveris OMR → manually check score → MusicXML → music21 → melody + harmony report → CSV/JSON`


## 1. Install dependencies (run once per Colab session)

In [ ]:
%pip -q install "music21>=9,<10" pymupdf pandas matplotlib

## 2. Import libraries and initialize the project

In [ ]:
from pathlib import Path
import json, re
import pandas as pd
import matplotlib.pyplot as plt
import fitz
from collections import Counter
from IPython.display import display, Markdown
from google.colab import files
from music21 import converter, stream, note, chord, harmony, key, roman, interval, meter, tempo, pitch

WORK = Path('/content/realbook_analysis')
WORK.mkdir(parents=True, exist_ok=True)
score = None
pdf_path = None
musicxml_path = None
print('Ready:', WORK)

## 3. Upload a PDF and preview the score image
Choose one song / lead-sheet PDF at a time. Prefer a clean 300-dpi scan with straight staff lines.

In [ ]:
incoming = files.upload()
pdfs = [name for name in incoming if name.lower().endswith('.pdf')]
if pdfs:
    pdf_path = WORK / Path(pdfs[0]).name
    pdf_path.write_bytes(incoming[pdfs[0]])
    doc = fitz.open(str(pdf_path))
    print(f'Uploaded {pdf_path.name}: {len(doc)} page(s)')
    for page_num in range(min(len(doc), 3)):
        page = doc[page_num]
        pix = page.get_pixmap(matrix=fitz.Matrix(1.5,1.5), alpha=False)
        display(__import__('PIL').Image.open(__import__('io').BytesIO(pix.tobytes('png'))))
    if len(doc)>3: print('Showing first 3 pages only.')
    doc.close()
else:
    print('No PDF uploaded. You can proceed using a MusicXML file or the demo score.')

## 4. Convert printed PDF to MusicXML (required OMR handoff)

**On your Windows PC** (recommended for easy error correction):

1. Install the open-source **Audiveris** OMR application from its official repository: https://github.com/Audiveris/audiveris .
2. Open the same PDF in Audiveris and run its recognition/transcription workflow.
3. **Inspect every bar.** Fix note names, rhythms, key signatures, chord symbols, measures, and repeats as needed. OMR is especially fallible on dense jazz lead sheets.
4. Select **Book → Export book** to produce `.mxl` (MusicXML).
5. Optionally open the exported file in MuseScore to confirm that it reads correctly, and save it again as MusicXML if you make corrections.
6. Return to Colab and upload the `.mxl` file in the next cell.

Helpful references: [Audiveris handbook](https://audiveris.github.io/audiveris/_pages/handbook/) · [Export MusicXML](https://audiveris.github.io/audiveris/_pages/tutorials/quick/export/)

**This is intentionally a semi-automated step**, rather than pretending that a PDF itself contains usable notes. A future version could install and invoke Audiveris in a controlled server environment, but still needs human verification.


## 5. Upload MusicXML, or use the built-in four-bar demonstration

In [ ]:
incoming_xml = files.upload()
xmls = [name for name in incoming_xml if Path(name).suffix.lower() in ('.mxl','.musicxml','.xml')]
if xmls:
    musicxml_path = WORK / Path(xmls[0]).name
    musicxml_path.write_bytes(incoming_xml[xmls[0]])
    score = converter.parse(str(musicxml_path))
    print('Loaded MusicXML:', musicxml_path.name)
else:
    print('No MusicXML supplied: creating a DEMO lead sheet (Cmaj7 | Am7 | Dm7 | G7).')
    melody = stream.Part(id='Melody'); melody.partName='Melody'
    melody.append(meter.TimeSignature('4/4'))
    melody.append(key.KeySignature(0))
    changes = [('Cmaj7',['E4','G4','B4','G4']),('Am7',['A4','C5','E5','C5']),('Dm7',['F4','A4','C5','A4']),('G7',['G4','B4','D5','F5'])]
    for symbol, pitches in changes:
        bar = stream.Measure(number=len(melody.getElementsByClass(stream.Measure))+1)
        bar.insert(0, harmony.ChordSymbol(symbol))
        for item in pitches: bar.append(note.Note(item,quarterLength=1))
        melody.append(bar)
    score=stream.Score(); score.insert(0,melody)
    print('Demo only: replace with verified MusicXML for real song analysis.')
print('Parts:', [(part.partName, len(list(part.recurse().notes))) for part in score.parts])

## 6. Extract melody notes, chord symbols, and measure numbers

The notebook uses **written chord symbols** (when present), not chord guesses made from the melody. For lead sheets, this distinction is important: a melody note is not necessarily a chord tone. Chord symbols must survive the OMR/export stage.


In [ ]:
part = max(score.parts, key=lambda p: sum(isinstance(x, note.Note) for x in p.recurse().notes)) if score.parts else score

def measure_number(el):
    m = el.getContextByClass(stream.Measure)
    return m.number if m and m.number is not None else None

melody_rows = []
for el in part.recurse().notes:
    if isinstance(el, note.Note):
        melody_rows.append(dict(measure=measure_number(el), pitch=el.pitch.nameWithOctave,
                                pitch_class=el.pitch.name, midi=el.pitch.midi,
                                duration_quarters=float(el.quarterLength),
                                offset_in_measure=float(el.getOffsetInHierarchy(el.getContextByClass(stream.Measure))) if el.getContextByClass(stream.Measure) else float(el.offset)))
    elif isinstance(el, chord.Chord):
        for pp in el.pitches:
            melody_rows.append(dict(measure=measure_number(el), pitch=pp.nameWithOctave,
                                    pitch_class=pp.name, midi=pp.midi,
                                    duration_quarters=float(el.quarterLength),
                                    offset_in_measure=float(el.offset)))

# Pull chord symbols across the score; exports sometimes place them in a separate part.
chord_rows = []
seen = set()
for obj in score.recurse().getElementsByClass(harmony.ChordSymbol):
    mnum = measure_number(obj)
    try: offset = float(obj.getOffsetInHierarchy(obj.getContextByClass(stream.Measure))) if obj.getContextByClass(stream.Measure) else float(obj.offset)
    except Exception: offset = float(obj.offset)
    figure = obj.figure
    identifier = (mnum, round(offset,5),figure)
    if identifier in seen: continue
    seen.add(identifier)
    chord_rows.append({'measure':mnum,'offset_in_measure':offset,'chord':figure,'object':obj})
chord_rows.sort(key=lambda r: (r['measure'] if r['measure'] is not None else -1, r['offset_in_measure']))

notes_df = pd.DataFrame(melody_rows)
chords_df = pd.DataFrame([{k:v for k,v in row.items() if k!='object'} for row in chord_rows], columns=['measure','offset_in_measure','chord'])
print(f'{len(notes_df)} note events · {len(chords_df)} chord symbols')
display(notes_df.head(20)); display(chords_df.head(20))
if chords_df.empty: print('WARNING: No chord symbols detected. Inspect your MusicXML and manually correct the OMR export.')

## 7. Analyze key, pitch inventory, melodic range, and intervals

In [ ]:
pitch_objects = [note.Note(p) for p in notes_df['pitch']] if not notes_df.empty else []
if pitch_objects:
    only_notes = stream.Stream(pitch_objects)
    guessed_key = only_notes.analyze('key')
    print('Estimated key from WRITTEN MELODY (not definitive):',guessed_key.tonic.name, guessed_key.mode)
    print('Pitch-class frequency:',dict(Counter(notes_df['pitch_class'])))
    low = pitch.Pitch(int(notes_df['midi'].min()))
    high = pitch.Pitch(int(notes_df['midi'].max()))
    print('Range:',low.nameWithOctave,'–',high.nameWithOctave)
    intervals = [interval.Interval(pitch.Pitch(a),pitch.Pitch(b)).directedName for a,b in zip(notes_df['pitch'][:-1],notes_df['pitch'][1:])]
    print('First melodic intervals:',intervals[:30])
else:
    guessed_key = None
    print('No pitched notes found.')


## 8. Analyze chord progression and Roman numerals
**Set the musical key manually if you know it.** Jazz tunes modulate, use secondary dominants, modal harmony, and borrowed chords. A one-key Roman-numeral pass is a first approximation, not a universal functional analysis.

In [ ]:
MANUAL_KEY = ''  # e.g. 'F major', 'D minor'; leave blank to use melody estimate

if MANUAL_KEY.strip():
    tokens = MANUAL_KEY.split()
    ANALYSIS_KEY = key.Key(tokens[0],tokens[1].lower() if len(tokens)>1 else 'major')
else:
    ANALYSIS_KEY = guessed_key

analysis = []
for r in chord_rows:
    symbol = r['object']
    entry = {'measure':r['measure'],'chord':r['chord']}
    try:
        entry['root'] = symbol.root().name if symbol.root() else None
        entry['pitch_classes'] = ', '.join(p.name for p in symbol.pitches)
    except Exception: entry['root'],entry['pitch_classes'] = None,None
    try: entry['roman_numeral'] = roman.romanNumeralFromChord(symbol,ANALYSIS_KEY).figure if ANALYSIS_KEY else None
    except Exception: entry['roman_numeral'] = None
    analysis.append(entry)
progression_df = pd.DataFrame(analysis, columns=['measure','chord','root','pitch_classes','roman_numeral'])
display(progression_df)
if progression_df.empty: print('Missing harmony data. Roman-numeral analysis needs recognized or manually supplied chord symbols.')

## 9. Compare each melody note to its accompanying chord
Chord-tone versus non-chord-tone labels are **descriptive**, not judgments of musical correctness. Extensions, suspensions, anticipation, and chromatic approach tones can be intentional. This first-pass check assumes chord changes occur within measures.

In [ ]:
comparison = []
for _, r in notes_df.iterrows():
    matches = [c for c in chord_rows if c['measure']==r['measure'] and c['offset_in_measure']<=r['offset_in_measure']+1e-6]
    active = matches[-1] if matches else None
    if active is None:
        # Chord symbols often carry across bars when not repeated.
        earlier = [c for c in chord_rows if c['measure'] is not None and r['measure'] is not None and c['measure']<r['measure']]
        active = earlier[-1] if earlier else None
    if active:
        symbol_pcs = {p.pitchClass for p in active['object'].pitches}
        label = 'chord tone' if int(r['midi'])%12 in symbol_pcs else 'non-chord tone / extension'
        name = active['chord']
    else: label,name='unknown (no chord)',None
    comparison.append({'measure':r['measure'],'melody_note':r['pitch'],'chord':name,'classification':label})
comparison_df = pd.DataFrame(comparison,columns=['measure','melody_note','chord','classification'])
display(comparison_df.head(60))

## 10. Save an analysis package (CSV + JSON)

In [ ]:
outdir = WORK/'results'; outdir.mkdir(exist_ok=True)
notes_df.to_csv(outdir/'melody_notes.csv',index=False)
chords_df.to_csv(outdir/'chord_symbols.csv',index=False)
progression_df.to_csv(outdir/'harmonic_analysis.csv',index=False)
comparison_df.to_csv(outdir/'melody_vs_chords.csv',index=False)
summary = {'musicxml_file':musicxml_path.name if musicxml_path else 'DEMO',
           'estimated_key_from_melody':str(guessed_key) if guessed_key else None,
           'analysis_key':str(ANALYSIS_KEY) if ANALYSIS_KEY else None,
           'note_events':len(notes_df),'chord_symbols':len(chords_df),
           'limitations':['PDF recognition performed externally via Audiveris','OMR needs manual review','single-key Roman numeral analysis is approximate']}
(outdir/'summary.json').write_text(json.dumps(summary,indent=2))
import shutil
archive = shutil.make_archive(str(WORK/'lead_sheet_analysis'),'zip',outdir)
print('Saved',archive)
files.download(archive)

## 11. Future upgrades (architecture)

- **PDF-to-MusicXML service:** deployment of Audiveris in a managed backend, with an explicit **verification/edit screen**.
- **Chord editor:** allow users to correct recognized symbols or enter one chord per beat/bar before analysis.
- **Key-center and modulation analysis:** analyze phrases, not only one global key.
- **Chord-scale suggestions:** jazz modes, tensions, avoid-note cautions, and alternative interpretations as **suggestions**, not certainties.
- **Music-theory explanations:** II–V–I, secondary dominant, tritone substitution, borrowed chords, turnarounds, resolutions.
- **Structured score export:** MusicXML and JSON become the app's canonical data format; preserve the original PDF as visual reference.
- **App UI:** after validating the notebook, use Gradio or a web front end for upload → preview → corrections → results.

**Note:** Repeats, first/second endings, pickup measures, multiple voices, and chord changes notated mid-measure need dedicated logic before treating this as a production-ready jazz analysis application.
